# Entrenamiento YOLOv8n-Pose — Giroscopio Dron
### Instrucciones
1. Menú → Entorno de ejecución → Cambiar tipo → **GPU T4**
2. Ejecutar las celdas **en orden**, una a una
3. En la celda 3 pegar tu API key de Roboflow
4. En la celda 3 pegar el nombre exacto de tu proyecto y versión

In [ ]:
# ============================================================
# CELDA 1 — Verificar GPU y instalar dependencias
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

!pip install ultralytics roboflow -q

import ultralytics
print('Ultralytics:', ultralytics.__version__)

import torch
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())

In [ ]:
# ============================================================
# CELDA 2 — Montar Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/giroscopio_proyecto'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Directorio de trabajo: {DRIVE_DIR}')

In [ ]:
# ============================================================
# CELDA 3 — Descargar dataset desde Roboflow
# ============================================================
# INSTRUCCIONES:
#   1. Ve a roboflow.com → tu proyecto → Versions
#   2. Haz clic en la versión → Export → Format: YOLOv8 Pose
#   3. Selecciona 'Show download code' y copia la API key
#   4. Rellena los tres valores de abajo

API_KEY      = "PEGA_AQUI_TU_API_KEY"       # ← tu clave de Roboflow
WORKSPACE    = "PEGA_AQUI_TU_WORKSPACE"      # ej: joss-workspace-7orjh
PROJECT      = "PEGA_AQUI_TU_PROYECTO"       # ej: giroscopio-dron
VERSION      = 1                              # número de versión

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download("yolov8", location="/content/dataset")

print('\nDataset descargado en /content/dataset')
import os
for split in ['train', 'valid', 'test']:
    img_dir = f'/content/dataset/{split}/images'
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f'  {split}: {n} imágenes')

In [ ]:
# ============================================================
# CELDA 4 — Verificar y corregir el data.yaml
# ============================================================
# Esta celda lee el yaml generado por Roboflow y se asegura
# de que tiene kpt_shape correcto para 9 keypoints

import yaml, os

yaml_path = '/content/dataset/data.yaml'

with open(yaml_path) as f:
    data = yaml.safe_load(f)

print('=== data.yaml ORIGINAL ===')
print(yaml.dump(data, default_flow_style=False))

# Corregir rutas absolutas (Roboflow a veces pone rutas relativas que fallan)
data['path']  = '/content/dataset'
data['train'] = 'train/images'
data['val']   = 'valid/images'
data['test']  = 'test/images'

# Asegurarse de que kpt_shape está bien para 9 keypoints
data['kpt_shape'] = [9, 3]

# nc debe ser 1 (solo clase 'giroscopio')
data['nc'] = 1
data['names'] = ['giroscopio']

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('=== data.yaml CORREGIDO ===')
with open(yaml_path) as f:
    print(f.read())

# Verificar que las rutas existen y tienen imágenes
print('=== VERIFICACIÓN DE RUTAS ===')
for split in ['train', 'valid']:
    img_path = f"/content/dataset/{split}/images"
    lbl_path = f"/content/dataset/{split}/labels"
    n_img = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
    n_lbl = len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0
    print(f"  {split}: {n_img} imágenes, {n_lbl} labels")
    if n_img == 0:
        print(f"  ⚠️  ERROR: no hay imágenes en {img_path}")

# Verificar formato de un label de muestra
print('\n=== MUESTRA DE LABEL (primera anotación) ===')
lbl_dir = '/content/dataset/train/labels'
sample = sorted(os.listdir(lbl_dir))[0]
with open(os.path.join(lbl_dir, sample)) as f:
    line = f.readline().strip()
values = line.split()
print(f'  Archivo: {sample}')
print(f'  Valores: {len(values)} (esperados: 5 bbox + 9×3 kp = 32)')
print(f'  Clase: {values[0]}')
print(f'  BBox (cx,cy,w,h): {values[1:5]}')
print(f'  Keypoints (primeros 3): {values[5:8]}')
if len(values) != 32:
    print(f'  ⚠️  ERROR: se esperan 32 valores, hay {len(values)}')
else:
    print(f'  ✓ Formato correcto')

In [ ]:
# ============================================================
# CELDA 5 — Entrenamiento
# ============================================================
from ultralytics import YOLO
import torch, os

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

model = YOLO('yolov8n-pose.pt')

results = model.train(
    data    = '/content/dataset/data.yaml',
    epochs  = 150,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    name    = 'giroscopio_v2',
    project = '/content/drive/MyDrive/giroscopio_proyecto/runs',
    # Augmentación — importante para generalizar con pocos datos
    hsv_h   = 0.015,   # variación de tono
    hsv_s   = 0.7,     # variación de saturación
    hsv_v   = 0.4,     # variación de brillo
    degrees = 15,      # rotación máxima
    flipud  = 0.3,     # volteo vertical
    fliplr  = 0.5,     # volteo horizontal
    scale   = 0.5,     # escala aleatoria
    patience= 30,      # early stopping si no mejora en 30 épocas
    save    = True,
    verbose = True,
)

print('\n=== RESULTADO DEL ENTRENAMIENTO ===')
print(f'mAP50:    {results.results_dict["metrics/mAP50(P)"]:.4f}')
print(f'mAP50-95: {results.results_dict["metrics/mAP50-95(P)"]:.4f}')
print(f'Mejor época: {results.best_fitness:.4f}')

In [ ]:
# ============================================================
# CELDA 6 — Verificar que el modelo aprendió ANTES de exportar
# ============================================================
import numpy as np
from ultralytics import YOLO

best_pt = '/content/drive/MyDrive/giroscopio_proyecto/runs/giroscopio_v2/weights/best.pt'
model = YOLO(best_pt)

# Test con imagen negra (no debe detectar nada)
img_black = np.zeros((640, 640, 3), dtype=np.uint8)
r1 = model(img_black, verbose=False)
print(f'Imagen negra → detecciones: {len(r1[0].boxes)}')

# Test con imagen de ruido (no debe superar 0.3 de confianza)
img_noise = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
r2 = model(img_noise, verbose=False)
conf_max = float(r2[0].boxes.conf.max()) if len(r2[0].boxes) else 0
print(f'Imagen ruido → detecciones: {len(r2[0].boxes)}, conf máx: {conf_max:.4f}')

# Leer una imagen real del dataset de validación y probarla
import os, random
val_imgs = os.listdir('/content/dataset/valid/images')
sample_img = os.path.join('/content/dataset/valid/images', random.choice(val_imgs))
r3 = model(sample_img, verbose=False)
conf_max_real = float(r3[0].boxes.conf.max()) if len(r3[0].boxes) else 0
print(f'Imagen real (val) → detecciones: {len(r3[0].boxes)}, conf máx: {conf_max_real:.4f}')

if conf_max_real > 0.3:
    print('\n✓ El modelo detecta el giroscopio. Proceder con la exportación.')
else:
    print('\n⚠️  El modelo NO detecta bien. Revisar el dataset antes de exportar.')

In [ ]:
# ============================================================
# CELDA 7 — Exportar a ONNX (formato correcto para Hailo)
# ============================================================
# IMPORTANTE: solo ejecutar si la celda 6 confirmó que el modelo funciona

from ultralytics import YOLO
import shutil, os

best_pt = '/content/drive/MyDrive/giroscopio_proyecto/runs/giroscopio_v2/weights/best.pt'
model = YOLO(best_pt)

# Exportar con los parámetros correctos para Hailo
# El ONNX resultante debe tener 9 tensores de salida SEPARADOS,
# NO un único output0 fusionado
onnx_path = model.export(
    format   = 'onnx',
    imgsz    = 640,
    opset    = 11,       # máxima compatibilidad con Hailo DFC
    simplify = True,
    dynamic  = False,
)

print(f'\nONNX exportado en: {onnx_path}')

# Verificar que el ONNX tiene el formato correcto (9 salidas separadas)
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path))
outputs = sess.get_outputs()
print(f'\n=== VERIFICACIÓN DEL ONNX ===')
print(f'Número de salidas: {len(outputs)}')
for o in outputs:
    print(f'  {o.name}  shape={o.shape}')

if len(outputs) == 1 and outputs[0].shape[-1] == 8400:
    print('\n⚠️  ONNX con salida fusionada — NO sirve para Hailo')
    print('   Solución: actualizar ultralytics y reexportar')
    print('   !pip install ultralytics --upgrade -q')
elif len(outputs) >= 9:
    print('\n✓ ONNX con 9 salidas separadas — correcto para Hailo')
else:
    print(f'\n? ONNX con {len(outputs)} salidas — verificar manualmente')

# Copiar a Drive
dest = '/content/drive/MyDrive/giroscopio_proyecto/giroscopio_pose_best_n_v2.onnx'
shutil.copy(str(onnx_path), dest)
print(f'\nONNX copiado a Drive: {dest}')

# También copiar el .pt a Drive
dest_pt = '/content/drive/MyDrive/giroscopio_proyecto/giroscopio_pose_best_n_v2.pt'
shutil.copy(best_pt, dest_pt)
print(f'.pt copiado a Drive: {dest_pt}')